# 02 — Чистка, сегменты, OSM-признаки, CatBoost travel-time

**Цель:** из сырых PLT-треков → чистый датасет сегментов → CatBoost-модель `segment_time_sec`

```
pip install catboost
```
(остальное уже есть в poetry-окружении)

In [ ]:
# !pip install catboost
# ! pip install scikit-learn

In [ ]:
import pickle
import ftplib, io, re, json, time, hashlib
from pathlib import Path
from datetime import datetime, timedelta
from math import radians, sin, cos, sqrt, atan2, degrees, atan

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.spatial import cKDTree

try:
    from catboost import CatBoostRegressor
    HAS_CB = True
except ImportError:
    HAS_CB = False
    print('catboost не установлен: pip install catboost')

try:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
    HAS_SKL = True
except ImportError:
    HAS_SKL = False
    print('scikit-learn не установлен: pip install scikit-learn')

DATA_DIR  = Path('data')
CACHE_DIR = Path('cache')
OSM_CACHE = CACHE_DIR / 'osm'
MODELS_DIR = Path('models')
for d in [DATA_DIR, CACHE_DIR, OSM_CACHE, MODELS_DIR]:
    d.mkdir(exist_ok=True)

FTP_HOST  = 'maps.lizaalert.ru'
FTP_PORT  = 21
FTP_USER  = 'lizaalert'
FTP_PASS  = '123'
OLE_EPOCH = datetime(1899, 12, 30)

print('OK')

## 1. Загрузка 200 миссий с FTP

In [ ]:
def ftp_connect():
    ftp = ftplib.FTP()
    ftp.connect(FTP_HOST, FTP_PORT, timeout=30)
    ftp.login(FTP_USER, FTP_PASS)
    ftp.set_pasv(True)
    return ftp

def list_dir(ftp, path):
    items, lines = [], []
    try:
        ftp.retrlines(f'LIST {path}', lines.append)
    except Exception:
        return items
    for line in lines:
        parts = line.split(None, 8)
        if len(parts) < 9:
            continue
        name = parts[8].strip()
        if name in ('.', '..'):
            continue
        is_dir = parts[0].startswith('d')
        size   = 0 if is_dir else int(parts[4])
        items.append((name, is_dir, size))
    return items

def download_file(ftp, remote, local: Path):
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.exists():
        return
    buf = io.BytesIO()
    ftp.retrbinary(f'RETR {remote}', buf.write)
    local.write_bytes(buf.getvalue())

N_MISSIONS = 200
ROOT = 'maps/tracks-archive'

# 1. Сначала собираем meta из уже скачанных файлов в data/
def collect_local(data_dir: Path):
    result = []
    for mdir in sorted(data_dir.iterdir()):
        if not mdir.is_dir() or not re.match(r'\d{4}-\d{2}-\d{2}', mdir.name):
            continue
        for plt_f in mdir.glob('*.plt'):
            result.append({'mission': mdir.name, 'tracker': plt_f.stem, 'path': plt_f})
    return result

local_meta     = collect_local(DATA_DIR)
local_missions = {r['mission'] for r in local_meta}
print(f'Уже локально: {len(local_missions)} миссий, {len(local_meta)} .plt файлов')

# 2. Один FTP-запрос: список 200 нужных миссий
ftp = ftp_connect()
all_missions = sorted(
    [i for i in list_dir(ftp, ROOT) if i[1] and re.match(r'\d{4}-\d{2}-\d{2}', i[0])],
    key=lambda x: x[0], reverse=True
)[:N_MISSIONS]
target_names = {m[0] for m in all_missions}
print(f'Целевые миссии: {all_missions[-1][0]} -> {all_missions[0][0]}')

# 3. Качаем только те, которых нет локально
need = [m for m in all_missions if m[0] not in local_missions]
print(f'Нужно докачать: {len(need)} миссий')

new_meta = []
for mi, (mname, _, _) in enumerate(need):
    plts = [i for i in list_dir(ftp, f'{ROOT}/{mname}')
            if not i[1] and i[0].lower().endswith('.plt')]
    for fname, _, _ in plts:
        local = DATA_DIR / mname / fname
        try:
            download_file(ftp, f'{ROOT}/{mname}/{fname}', local)
            new_meta.append({'mission': mname, 'tracker': Path(fname).stem, 'path': local})
        except Exception:
            pass
    if need and (mi + 1) % 10 == 0:
        print(f'  {mi+1}/{len(need)} новых миссий...')

ftp.quit()

# 4. Итоговый meta - только миссии из топ-200
meta = [r for r in local_meta if r['mission'] in target_names] + new_meta
print(f'\nИтого .plt файлов: {len(meta)}')
print(f'Поисков в выборке: {len({r["mission"] for r in meta})}')

## 2. Парсинг PLT + чистка на уровне трека

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

def parse_plt(path: Path) -> pd.DataFrame:
    try:
        text = path.read_text(encoding='latin-1', errors='ignore')
    except Exception:
        return pd.DataFrame()
    lines = text.splitlines()
    data_start = 6
    for i in range(3, min(10, len(lines))):
        if re.match(r'^\s*\d+\s*$', lines[i]):
            data_start = i + 1
            break
    records = []
    for line in lines[data_start:]:
        parts = line.split(',')
        if len(parts) < 5:
            continue
        try:
            lat = float(parts[0])
            lon = float(parts[1])
            if abs(lat) < 1 or abs(lon) < 1:
                continue
            new_seg = int(parts[2])
            alt_ft  = float(parts[3])
            alt_m   = None if alt_ft < -700 else round(alt_ft * 0.3048, 1)
            ts      = None
            ole     = float(parts[4])
            if ole > 30000:
                ts = OLE_EPOCH + timedelta(days=ole)
            records.append({'lat': lat, 'lon': lon, 'new_seg': new_seg,
                            'alt_m': alt_m, 'ts': ts})
        except Exception:
            continue
    return pd.DataFrame(records)

# Пороги чистки на уровне трека
MIN_POINTS   = 10
MIN_DIST_KM  = 0.3
MAX_DIST_KM  = 300.0

good_tracks = []
skipped = {'short': 0, 'no_ts': 0, 'too_long': 0, 'error': 0}

for rec in meta:
    try:
        df = parse_plt(rec['path'])
    except Exception:
        skipped['error'] += 1
        continue
    if len(df) < MIN_POINTS:
        skipped['short'] += 1
        continue
    if df['ts'].isna().mean() > 0.5:
        skipped['no_ts'] += 1
        continue
    pts = df[['lat','lon']].values
    dist_km = sum(haversine_m(*pts[i], *pts[i+1]) for i in range(len(pts)-1)) / 1000
    if dist_km < MIN_DIST_KM:
        skipped['short'] += 1
        continue
    if dist_km > MAX_DIST_KM:
        skipped['too_long'] += 1
        continue
    df['mission']  = rec['mission']
    df['tracker']  = rec['tracker']
    df['track_id'] = f"{rec['mission']}__{rec['tracker']}"
    good_tracks.append(df)

tracks = pd.concat(good_tracks, ignore_index=True)
print(f'Принято треков:  {len(good_tracks)}')
print(f'Отброшено:       {skipped}')
print(f'Всего точек:     {len(tracks):,}')
print(f'С timestamp:     {tracks["ts"].notna().sum():,} ({100*tracks["ts"].notna().mean():.1f}%)')
print(f'С высотой:       {tracks["alt_m"].notna().sum():,} ({100*tracks["alt_m"].notna().mean():.1f}%)')

## 3. Построение сегментов


In [ ]:
seg_records = []

for track_id, grp in tracks.groupby('track_id'):
    grp = (grp.sort_values('ts') if grp['ts'].notna().any()
           else grp).reset_index(drop=True)
    for i in range(len(grp) - 1):
        r1, r2 = grp.iloc[i], grp.iloc[i + 1]
        if r2['new_seg'] == 1:
            continue
        dist_m = haversine_m(r1.lat, r1.lon, r2.lat, r2.lon)
        if dist_m < 0.5:
            continue
        dt_s = None
        if r1['ts'] is not None and r2['ts'] is not None:
            dt_s = (r2['ts'] - r1['ts']).total_seconds()
            if dt_s <= 0:
                dt_s = None
        elev_diff = (round(r2['alt_m'] - r1['alt_m'], 1)
                     if r1['alt_m'] is not None and r2['alt_m'] is not None
                     else None)
        seg_records.append({
            'track_id':    track_id,
            'mission':     r1['mission'],
            'lat1': r1.lat, 'lon1': r1.lon,
            'lat2': r2.lat, 'lon2': r2.lon,
            'lat_mid': (r1.lat + r2.lat) / 2,
            'lon_mid': (r1.lon + r2.lon) / 2,
            'dist_m':      round(dist_m, 2),
            'dt_s':        dt_s,
            'elev_diff_m': elev_diff,
        })

segs = pd.DataFrame(seg_records)
print(f'Сегментов до фильтрации: {len(segs):,}')
print(segs[['dist_m','dt_s','elev_diff_m']].describe().round(2))


## 4. Фильтрация сегментов + классификация треков

In [ ]:
SEG_DIST_MIN  = 1.0
SEG_DIST_MAX  = 500.0
SEG_DT_MIN    = 1.0
SEG_DT_MAX    = 3600.0
SEG_SPEED_MAX = 25.0
SEG_SPEED_MIN = 0.05

PED_SPEED_THRESH = 8.0

n_before = len(segs)
sf = segs.copy()
sf = sf[sf['dist_m'].between(SEG_DIST_MIN, SEG_DIST_MAX)]
sf = sf[sf['dt_s'].notna() & sf['dt_s'].between(SEG_DT_MIN, SEG_DT_MAX)]
sf['speed_kmh'] = sf['dist_m'] / sf['dt_s'] * 3.6
sf = sf[sf['speed_kmh'].between(SEG_SPEED_MIN, SEG_SPEED_MAX)]

track_med = sf.groupby('track_id')['speed_kmh'].median()
ped_tracks = track_med[track_med <= PED_SPEED_THRESH].index
sf['is_pedestrian'] = sf['track_id'].isin(ped_tracks)

print(f'До фильтрации:    {n_before:,}')
print(f'После фильтрации: {len(sf):,}')
print(f'Пешие сегменты:   {sf["is_pedestrian"].sum():,} ({100*sf["is_pedestrian"].mean():.1f}%)')
print(f'Пеших треков:     {sf[sf["is_pedestrian"]]["track_id"].nunique()}')
print()
ped = sf[sf['is_pedestrian']]['speed_kmh']
print(f'Скорость пеших - медиана: {ped.median():.2f} км/ч')
print(f'10-90 перцентили:         {ped.quantile(0.1):.2f} - {ped.quantile(0.9):.2f} км/ч')


## 5. Уклон из GPS-высоты

PLT уже содержит `alt_ft` для каждой точки — используем без внешних DEM.
`slope_deg = arctan(|Δh| / dist_m) × 180 / π`

In [ ]:
def slope_deg(elev_diff_m, dist_m):
    if elev_diff_m is None or dist_m < 1:
        return None
    return round(degrees(atan(abs(elev_diff_m) / max(dist_m, 0.1))), 3)

sf = sf.copy()
sf['slope_deg'] = sf.apply(
    lambda r: slope_deg(r['elev_diff_m'], r['dist_m']), axis=1
)
sf['going_up'] = (sf['elev_diff_m'].fillna(0) > 0).astype(int)

has = sf['slope_deg'].notna()
print(f'Сегментов с уклоном: {has.sum():,} ({100*has.mean():.1f}%)')
if has.any():
    s = sf.loc[has, 'slope_deg']
    print(f'  медиана: {s.median():.1f}deg  90p: {s.quantile(0.9):.1f}deg  max: {s.max():.1f}deg')

## 5b. Фильтр городских поисков

Городской поиск (поиск в квартале / по домам) не подходит для terrain-модели:
нет рельефа, нет естественной местности, треки идут строго по улицам.

Детектируем по двум признакам на чистых GPS-данных:
- **Flat**: медианный уклон трека < 1.5°
- **Grid**: >45% сегментов направлены близко к 0° / 90° / 180° / 270° (±15°)

In [ ]:
# Heading каждого сегмента (atan2 в градусах, 0-360)
_dlat = sf['lat2'] - sf['lat1']
_dlon = sf['lon2'] - sf['lon1']
sf = sf.copy()
sf['heading_deg'] = np.degrees(np.arctan2(_dlon, _dlat)) % 360

# Per-track: доля сегментов близко к кардинальным (0/90/180/270) +/-15deg
_CARDINALS = np.array([0, 90, 180, 270, 360])

def _grid_frac(h_series):
    h = h_series.values % 360
    return (np.min(np.abs(h[:, None] - _CARDINALS), axis=1) < 15).mean()

def _slope_med(s_series):
    s = s_series.dropna()
    return s.median() if len(s) else np.nan

_tstats = sf.groupby('track_id').agg(
    grid_frac  = ('heading_deg', _grid_frac),
    slope_med  = ('slope_deg',   _slope_med),
)

FLAT_THRESH = 1.5   # deg     - плоский трек
GRID_THRESH = 0.45  # доля  - движение по сетке улиц

_urban = _tstats[
    (_tstats['slope_med'] < FLAT_THRESH) &
    (_tstats['grid_frac'] > GRID_THRESH)
].index

sf['is_urban'] = sf['track_id'].isin(_urban)

n_urban = sf[sf['is_urban']]['track_id'].nunique()
n_total = sf['track_id'].nunique()
print(f'Городских треков: {n_urban} из {n_total} ({100*n_urban/n_total:.1f}%)')
print(f'Сегментов до фильтра: {len(sf):,}')

sf = sf[~sf['is_urban']].copy()
print(f'Сегментов после:      {len(sf):,}')
print(f'Пеших сегментов:      {sf["is_pedestrian"].sum():,}')

# Сводка по удалённым миссиям (для проверки)
removed_missions = sf.loc[sf['track_id'].isin(_urban), 'mission'].unique() if len(_urban) else []
# (sf уже без городских, смотрим через _tstats)
_urban_missions = sf.assign(is_urban=sf['track_id'].isin(_urban))  # уже пусто
# Покажем названия через original stats
_urban_mission_list = sf.copy()  # sf уже отфильтрован
# Проще: выводим из track_id
sample_urban = sorted(_urban)[:10]
if sample_urban:
    print('\nПример удалённых треков:')
    for t in sample_urban:
        mission = t.split('__')[0]
        s = _tstats.loc[t]
        print(f'  {mission:<40} slope={s.slope_med:.1f}deg  grid={s.grid_frac:.2f}')

In [ ]:
# DEM slope вместо GPS slope
# GPS вертикальная точность +/-10-30м - для сегмента 13м это чистый шум.
# COP30 DEM: +/-1-3м -> slope намного точнее + нет domain shift с продовой моделью.

import rasterio, rasterio.transform
from rasterio.merge import merge as _rmerge
from math import floor, radians, cos, degrees, atan

DEM_TILE_DIR = Path('/Users/antonmikhaylyuk/Documents/magister_final_proj'
                    '/drone_code/data/terrain/tile_cache/dem')
DEM_SLOPE_CACHE = CACHE_DIR / 'dem_slope.pkl'

def _tile_path(lat, lon):
    tl = floor(lat); to = floor(lon)
    ls = f'N{tl:02d}' if tl >= 0 else f'S{abs(tl):02d}'
    os_ = f'E{to:03d}' if to >= 0 else f'W{abs(to):03d}'
    return DEM_TILE_DIR / f'COP30_{ls}{os_}.tif'

def _dem_slope_for_mission(lats, lons):
    """Возвращает массив slope_deg для точек (lats, lons) из COP30 тайлов."""
    needed = set()
    for la, lo in zip(lats, lons):
        needed.add(_tile_path(la, lo))
    paths = [str(p) for p in needed if p.exists()]
    if not paths:
        return np.full(len(lats), np.nan)

    pad = 0.02
    bounds = (min(lons)-pad, min(lats)-pad, max(lons)+pad, max(lats)+pad)
    datasets = [rasterio.open(p) for p in paths]
    try:
        mosaic, tf = _rmerge(datasets, bounds=bounds)
    finally:
        for ds in datasets: ds.close()

    z = mosaic[0].astype(np.float32)
    nodata = datasets[0].nodata
    if nodata is not None:
        z = np.where(z == nodata, np.nan, z)

    # slope из градиента растра
    lat_c = (min(lats) + max(lats)) / 2
    dx_m = abs(tf.a) * 111320 * cos(radians(lat_c))
    dy_m = abs(tf.e) * 111320
    dz_dy, dz_dx = np.gradient(z, dy_m, dx_m)
    slope_arr = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
    slope_arr = np.where(np.isnan(z), np.nan, slope_arr)

    # векторный lookup
    rows, cols = rasterio.transform.rowcol(tf, list(lons), list(lats))
    rows = np.array(rows); cols = np.array(cols)
    h, w = slope_arr.shape
    valid = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    result = np.full(len(lats), np.nan)
    result[valid] = slope_arr[rows[valid], cols[valid]]
    return result

# --- цикл по миссиям ---
if DEM_SLOPE_CACHE.exists():
    with open(DEM_SLOPE_CACHE, 'rb') as _f:
        _cached = pickle.load(_f)
    if len(_cached) == len(sf):
        sf['slope_deg'] = _cached
        print(f'DEM slope загружен из кэша ({sf["slope_deg"].notna().mean()*100:.1f}% покрытие)')
        raise SystemExit('cached')  # прыгаем через вычисление

sf_r = sf.reset_index(drop=True)
dem_slope = np.full(len(sf_r), np.nan)
missions_uniq = sf_r['mission'].unique()
errors = []

for mi, mission in enumerate(missions_uniq):
    mask = (sf_r['mission'] == mission).values
    idxs = np.where(mask)[0]
    mdf  = sf_r.iloc[idxs]
    try:
        vals = _dem_slope_for_mission(mdf['lat_mid'].values, mdf['lon_mid'].values)
        dem_slope[idxs] = vals
    except Exception as e:
        errors.append(f'{mission}: {e}')
    if (mi+1) % 30 == 0 or mi == len(missions_uniq)-1:
        ok = (~np.isnan(dem_slope)).sum()
        print(f'  {mi+1}/{len(missions_uniq)}  покрыто: {ok:,} ({100*ok/len(sf_r):.1f}%)')

sf = sf_r.copy()

# Заменяем GPS slope на DEM где есть тайл, иначе оставляем GPS
has_dem = ~np.isnan(dem_slope)
sf.loc[has_dem, 'slope_deg'] = dem_slope[has_dem]
# going_up оставляем от GPS - он даёт направление (вверх/вниз)

with open(DEM_SLOPE_CACHE, 'wb') as _f:
    pickle.dump(sf['slope_deg'].values, _f)

print(f'DEM slope: {has_dem.sum():,} / {len(sf):,} ({100*has_dem.mean():.1f}%)')
gps_med = sf.loc[~has_dem, 'slope_deg'].median()
dem_med = sf.loc[has_dem, 'slope_deg'].median()
print(f'GPS slope медиана: {gps_med:.2f}deg   DEM slope медиана: {dem_med:.2f}deg')
if errors:
    print(f'[!] {len(errors)} миссий без DEM: {errors[:2]}')
print('Важно: миссии без кэшированных тайлов остаются с GPS slope.')
print('Чтобы скачать тайлы - поставь OPENTOPO_API_KEY и запусти prefetch.')


## 6. OSM-признаки

Для каждой миссии: один запрос к Overpass API (французское зеркало, fallback немецкое),
pickle-кэш в `cache/osm/`.
Результат: `on_road`, `on_trail`, `near_water` — расстояния до ближайшего объекта (KDTree).


In [ ]:
import pickle as _pk

# Французское зеркало надёжнее для пакетных запросов; немецкое - fallback
_MIRRORS = [
    "https://overpass.openstreetmap.fr/api/interpreter",
    "https://overpass-api.de/api/interpreter",
]
_HEADERS = {"User-Agent": "sar-drone/0.1 (thesis project)", "Accept": "*/*"}

TRAIL_HW = {'path','footway','track','bridleway','cycleway','steps'}

def bbox_wsne(df_rows):
    pad = 0.025
    return (
        df_rows['lon_mid'].min() - pad,
        df_rows['lat_mid'].min() - pad,
        df_rows['lon_mid'].max() + pad,
        df_rows['lat_mid'].max() + pad,
    )

def _overpass_query(west, south, east, north, timeout=45):
    """Один запрос: дороги + тропы + вода."""
    q = f"""
[out:json][timeout:{timeout}];
(
  way["highway"]({south},{west},{north},{east});
  way["waterway"]({south},{west},{north},{east});
  way["natural"="water"]({south},{west},{north},{east});
);
out geom;
"""
    for url in _MIRRORS:
        try:
            r = requests.post(url, data={'data': q}, headers=_HEADERS,
                              timeout=timeout + 10)
            if r.status_code == 200:
                return r.json()
        except Exception as e:
            print(f'  [mirror] {url.split("/")[2]}: {e}')
    return None

def _parse_response(data):
    roads, trails, water = [], [], []
    for el in (data or {}).get('elements', []):
        if el.get('type') != 'way':
            continue
        coords = [(n['lat'], n['lon']) for n in el.get('geometry', []) if 'lat' in n]
        if not coords:
            continue
        tags = el.get('tags', {})
        hw = tags.get('highway', '')
        if 'waterway' in tags or tags.get('natural') == 'water':
            water.extend(coords)
        elif hw in TRAIL_HW:
            trails.extend(coords)
        elif hw:
            roads.extend(coords)
    to_np = lambda l: np.array(l, dtype=np.float32) if l else np.zeros((0,2), np.float32)
    return to_np(roads), to_np(trails), to_np(water)

def scaled_kdt(pts, cos_lat):
    if len(pts) == 0:
        return None
    return cKDTree(np.column_stack([pts[:,0]*111320, pts[:,1]*111320*cos_lat]))

def query_dist(kdt, pts_sc):
    if kdt is None or len(pts_sc) == 0:
        return np.full(len(pts_sc), np.inf)
    d, _ = kdt.query(pts_sc, workers=-1)
    return d

def fetch_and_cache(mission, bbox_wsne_val):
    ckey = hashlib.md5(mission.encode()).hexdigest()[:12]
    cf = OSM_CACHE / f'{ckey}.pkl'
    if cf.exists():
        with open(cf, 'rb') as f:
            return _pk.load(f)
    west, south, east, north = bbox_wsne_val
    data = _overpass_query(west, south, east, north)
    result = _parse_response(data)
    with open(cf, 'wb') as f:
        _pk.dump(result, f)
    return result

print('OSM functions ready (fr mirror + combined query)')

In [ ]:
missions_uniq = sf['mission'].unique()
already = sum(1 for m in missions_uniq
              if (OSM_CACHE / f'{hashlib.md5(m.encode()).hexdigest()[:12]}.pkl').exists())
need_fetch = len(missions_uniq) - already
print(f'Миссий: {len(missions_uniq)}  кэш: {already}  нужно скачать: {need_fetch}  (~{need_fetch*6//60} мин)')

road_d  = np.full(len(sf), np.inf)
trail_d = np.full(len(sf), np.inf)
water_d = np.full(len(sf), np.inf)
sf_idx  = sf.reset_index(drop=True)
errors  = []
fetched = 0
t0 = time.time()

for mi, mission in enumerate(missions_uniq):
    mask = sf_idx['mission'] == mission
    idxs = np.where(mask.values)[0]
    mdf  = sf_idx[mask]

    bbox     = bbox_wsne(mdf)
    mean_lat = mdf['lat_mid'].mean()
    cos_lat  = cos(radians(mean_lat))

    ckey       = hashlib.md5(mission.encode()).hexdigest()[:12]
    was_cached = (OSM_CACHE / f'{ckey}.pkl').exists()

    try:
        rp, tp, wp = fetch_and_cache(mission, bbox)
    except Exception as e:
        errors.append(f'{mission}: {e}')
        continue

    if not was_cached:
        fetched += 1
        if fetched < need_fetch:
            elapsed = time.time() - t0
            eta_min = (elapsed / fetched) * (need_fetch - fetched) / 60
            print(f'  [{mi+1}/{len(missions_uniq)}] {mission[:35]:<35}  ETA ~{eta_min:.0f} мин')
        time.sleep(5)  # вежливая пауза (5с достаточно при fr-зеркале)

    kd_r = scaled_kdt(rp, cos_lat)
    kd_t = scaled_kdt(tp, cos_lat)
    kd_w = scaled_kdt(wp, cos_lat)

    mids = np.column_stack([mdf['lat_mid'].values * 111320,
                             mdf['lon_mid'].values * 111320 * cos_lat])

    road_d[idxs]  = query_dist(kd_r, mids)
    trail_d[idxs] = query_dist(kd_t, mids)
    water_d[idxs] = query_dist(kd_w, mids)

sf_idx['dist_road_m']  = road_d.clip(0, 9999)
sf_idx['dist_trail_m'] = trail_d.clip(0, 9999)
sf_idx['dist_water_m'] = water_d.clip(0, 9999)
sf_idx['on_road']    = (sf_idx['dist_road_m']  < 30).astype(int)
sf_idx['on_trail']   = (sf_idx['dist_trail_m'] < 30).astype(int)
sf_idx['near_water'] = (sf_idx['dist_water_m'] < 50).astype(int)
sf = sf_idx

if errors:
    print(f'[!] Пропущено {len(errors)} миссий: {errors[:3]}')
print(f'\non_road:    {sf["on_road"].mean()*100:.1f}%')
print(f'on_trail:   {sf["on_trail"].mean()*100:.1f}%')
print(f'near_water: {sf["near_water"].mean()*100:.1f}%')
print(f'Готово за {(time.time()-t0)/60:.1f} мин')


## 7. Финальный датасет

In [ ]:
HAS_OSM_FEATS = 'on_road' in sf.columns

final = sf[sf['is_pedestrian']].copy()

# --- Feature engineering ---
final['log_dist']     = np.log1p(final['dist_m'])          # log масштаб расстояния
final['slope_sq']     = final['slope_deg'] ** 2            # нелинейный эффект рельефа
final['slope_x_road'] = final['slope_deg'] * final.get('on_road', 0)  # взаимодействие

BASE_FEATURES = ['log_dist', 'slope_deg', 'slope_sq', 'elev_diff_m', 'going_up']
OSM_FEATURES  = ['on_road', 'on_trail', 'near_water', 'slope_x_road']
FEATURES = BASE_FEATURES + (OSM_FEATURES if HAS_OSM_FEATS else [])
TARGET   = 'speed_kmh'

print('OSM-признаки:', 'ЕСТЬ' if HAS_OSM_FEATS else 'НЕТ')
print(f'Всего признаков: {len(FEATURES)}: {FEATURES}')

model_df = final.dropna(subset=FEATURES + [TARGET]).copy()
v1, v99  = model_df[TARGET].quantile([0.01, 0.99])
model_df = model_df[model_df[TARGET].between(v1, v99)]

print(f'\nПеших сегментов: {len(final):,}  -> для модели: {len(model_df):,}')
print(f'Треков: {model_df["track_id"].nunique()}  Миссий: {model_df["mission"].nunique()}')
print(f'\nTarget speed_kmh: медиана {model_df[TARGET].median():.2f}  std {model_df[TARGET].std():.2f}')

import pickle as _pk
with open(CACHE_DIR / 'segments_model.pkl', 'wb') as f:
    _pk.dump(model_df, f)


## 8. Визуализация чистых данных

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f'Чистые пешие сегменты  ({len(final):,} шт., {final["track_id"].nunique()} треков)', fontsize=13)

# 1. Скорость
ax = axes[0,0]
spd = final['speed_kmh'].clip(0, 15)
spd.hist(bins=60, ax=ax, color='#3b82f6', edgecolor='none')
ax.axvline(spd.median(), color='red', ls='--', lw=1.5,
           label=f'медиана {spd.median():.2f} км/ч')
ax.set_title('Скорость сегмента (км/ч)'); ax.legend(fontsize=8)

# 2. Длина сегмента
ax = axes[0,1]
final['dist_m'].clip(0, 300).hist(bins=60, ax=ax, color='#22c55e', edgecolor='none')
ax.axvline(final['dist_m'].median(), color='red', ls='--', lw=1.5,
           label=f'медиана {final["dist_m"].median():.0f} м')
ax.set_title('Длина сегмента (м)'); ax.legend(fontsize=8)

# 3. Уклон
ax = axes[0,2]
sl = final['slope_deg'].dropna().clip(0, 45)
if len(sl):
    sl.hist(bins=60, ax=ax, color='#f97316', edgecolor='none')
    ax.axvline(sl.median(), color='red', ls='--', lw=1.5,
               label=f'медиана {sl.median():.1f}deg')
    ax.legend(fontsize=8)
ax.set_title('Уклон (deg)')

# 4. Время сегмента (target)
ax = axes[1,0]
t = final[TARGET].clip(0, 300)
t.hist(bins=60, ax=ax, color='#a855f7', edgecolor='none')
ax.axvline(t.median(), color='red', ls='--', lw=1.5,
           label=f'медиана {t.median():.0f} с')
ax.set_title('Время сегмента (с)'); ax.legend(fontsize=8)

# 5. Скорость vs уклон
ax = axes[1,1]
sc = final.dropna(subset=['slope_deg']).sample(min(20000, len(final)), random_state=42)
ax.scatter(sc['slope_deg'].clip(0, 40), sc['speed_kmh'].clip(0, 12),
           s=0.4, alpha=0.15, c='#ef4444', rasterized=True)
ax.set_xlabel('Уклон (deg)'); ax.set_ylabel('Скорость (км/ч)')
ax.set_title('Скорость vs Уклон')

# 6. GPS-точки на карте
ax = axes[1,2]
smap = final.sample(min(60000, len(final)), random_state=42)
ax.scatter(smap['lon_mid'], smap['lat_mid'], s=0.2, alpha=0.1,
           c='#3b82f6', rasterized=True)
ax.set_title(f'GPS midpoints (sample {min(60000,len(final)):,})')
ax.set_xlabel('lon'); ax.set_ylabel('lat')
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('figures/segments_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: figures/segments_eda.png')

## 9. CatBoost — GroupKFold CV + финальная модель

**Таргет:** `speed_kmh`.
**CV:** GroupKFold(5) по `mission` — честная оценка без утечки между треками одной миссии.
**Признаки:** базовые GPS + OSM + инженерные (`log_dist`, `slope²`, `slope×road`).


In [ ]:
from sklearn.model_selection import GroupKFold

X = model_df[FEATURES].copy()
y = model_df[TARGET].copy()
groups = model_df['mission'].values

for col in FEATURES:
    if X[col].isna().any():
        X[col] = X[col].fillna(X[col].median())

CAT_FEATURES = ['going_up']
if HAS_OSM_FEATS:
    CAT_FEATURES += ['on_road', 'on_trail', 'near_water']
for col in CAT_FEATURES:
    X[col] = X[col].astype(int)

CB_PARAMS = dict(
    iterations=800, learning_rate=0.05, depth=6,
    loss_function='MAE', eval_metric='MAE',
    random_seed=42, verbose=0,
)

# --- GroupKFold CV ---
gkf = GroupKFold(n_splits=5)
cv_mae, cv_r2 = [], []

print('GroupKFold CV (5 folds по миссиям):')
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr,  X_val  = X.iloc[tr_idx],  X.iloc[val_idx]
    y_tr,  y_val  = y.iloc[tr_idx],  y.iloc[val_idx]

    m = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
    m.fit(X_tr, y_tr, cat_features=CAT_FEATURES,
          eval_set=(X_val, y_val), use_best_model=True)

    yp = m.predict(X_val)
    fold_mae = mean_absolute_error(y_val, yp)
    fold_r2  = r2_score(y_val, yp)
    cv_mae.append(fold_mae)
    cv_r2.append(fold_r2)
    n_miss_val = len(set(groups[val_idx]))
    print(f'  Fold {fold+1}: MAE={fold_mae:.3f} км/ч  R^2={fold_r2:.3f}  ({n_miss_val} миссий в val)')

print(f'\nCV итог:  MAE = {np.mean(cv_mae):.3f} +/- {np.std(cv_mae):.3f} км/ч')
print(f'          R^2  = {np.mean(cv_r2):.3f} +/- {np.std(cv_r2):.3f}')

# --- Финальная модель на всех данных ---
print('\nОбучаю финальную модель на полных данных...')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = CatBoostRegressor(**CB_PARAMS, early_stopping_rounds=50)
model.fit(X_train, y_train, cat_features=CAT_FEATURES,
          eval_set=(X_test, y_test), use_best_model=True)
print('Готово.')


## 10. Оценка модели

In [ ]:
if HAS_CB and 'model' in dir():
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2   = r2_score(y_test, y_pred)

    y_base_const  = np.full(len(y_test), 3.5)
    y_base_median = np.full(len(y_test), float(y_train.median()))
    import math
    slope_rad = X_test['slope_deg'].apply(math.radians)
    y_base_tobler = 6 * np.exp(-3.5 * np.abs(slope_rad + 0.05))

    mae_b1 = mean_absolute_error(y_test, y_base_const)
    mae_b2 = mean_absolute_error(y_test, y_base_median)
    mae_b3 = mean_absolute_error(y_test, y_base_tobler)

    print('=== РЕЗУЛЬТАТЫ (hold-out 20%) ===')
    print(f'CatBoost   MAE={mae:.3f} км/ч  RMSE={rmse:.3f}  R^2={r2:.3f}')
    print(f'Baseline   3.5 km/h const:    MAE={mae_b1:.3f}')
    print(f'Baseline   train median:       MAE={mae_b2:.3f}')
    print(f'Baseline   Tobler formula:     MAE={mae_b3:.3f}')
    print(f'Улучшение vs 3.5 км/ч: {(mae_b1-mae)/mae_b1*100:.1f}%')
    print(f'Улучшение vs Tobler:   {(mae_b3-mae)/mae_b3*100:.1f}%')
    print()
    print(f'CV R^2 (честный, по миссиям): {np.mean(cv_r2):.3f} +/- {np.std(cv_r2):.3f}')
    print(f'Hold-out R^2 (random split):  {r2:.3f}  <- вероятно завышен из-за корреляции треков')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ax = axes[0]
    fi = pd.DataFrame({'feature': FEATURES,
                       'importance': model.get_feature_importance()
                      }).sort_values('importance')
    ax.barh(fi['feature'], fi['importance'], color='#3b82f6')
    ax.set_title('Feature Importance')
    for i, (_, row) in enumerate(fi.iterrows()):
        ax.text(row['importance'] + 0.2, i, f"{row['importance']:.1f}", va='center', fontsize=8)

    ax = axes[1]
    sidx = np.random.default_rng(0).choice(len(y_test), min(3000, len(y_test)), replace=False)
    ax.scatter(np.array(y_test)[sidx], y_pred[sidx], s=2, alpha=0.3, c='#ef4444', rasterized=True)
    lim = max(np.array(y_test).max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set_xlabel('Факт (км/ч)'); ax.set_ylabel('Предсказание (км/ч)')
    ax.set_title(f'Predicted vs Actual speed  (R^2={r2:.3f})')

    plt.tight_layout()
    plt.savefig('figures/model_eval.png', dpi=150, bbox_inches='tight')
    plt.show()

    model.save_model(str(MODELS_DIR / 'travel_time.cbm'))
    print(f'\nМодель сохранена: models/travel_time.cbm')
    print('Применение: time_sec = dist_m / (model.predict([features]) / 3.6)')
